# 02 - Parametric simulation: ACCIM predefined model

This notebook applies the same parametric logic using ACCIM predefined models, focusing on clean parameter setup and reproducible execution.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `AccimPredefModelsParamSim`
- `AccimPredefModelsParamSim.select_outputs`
- `AccimPredefModelsParamSim.set_problem`
- `AccimPredefModelsParamSim.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.apply_outputs_preflight`
- `parametric.set_output_readers`
- `parametric.get_available_parameters`
- `parametric.set_parameters`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## ACCIM predefined parametric: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [ ]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import AccimPredefModelsParamSim

pd.set_option("display.max_columns", 160)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## ACCIM predefined parametric: define paths and execution folders

This block defines supporting paths and run constants used by later cells.

Centralizing these values reduces hidden coupling and keeps the execution flow easier to audit.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_file`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_file`, `epw_list`, `RESULTS_DIRNAME`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_file = "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf"
epw_list = [
	"Seville.epw",
	"Sydney.epw",
]
RESULTS_DIRNAME = "nb02_parametric_predefined_results"

print("IDF exists:", (PROJECT_ROOT / idf_file).exists())
print("EPWs exist:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## ACCIM predefined parametric: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [ ]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


## ACCIM predefined parametric: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.
- `building`: loaded IDF model used in this workflow.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`, `parametric`.

In [ ]:
building = ef.get_building(idf_file)
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"

parametric = AccimPredefModelsParamSim(
	building=building,
	epws=epw_list,
	output_freqs=["Hourly"],
)


## ACCIM predefined parametric: run output preflight checks before execution

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.
- `df_output_meter`: final meter table used by the simulation.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
df_meters_sel = pd.DataFrame(
	{
		"key_name": objective_meters,
		"frequency": ["Hourly"] * len(objective_meters),
	}
)

parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=None,
	clean_mode="all",
	validate_before_apply=False,
	validate_after_apply=True,
	on_missing="raise",
)
parametric.set_output_readers(df_output_meter=df_meters_sel)
print("Objective meters configured:", list(df_meters_sel["key_name"]))


## ACCIM predefined parametric: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

The study compares the three categories of EN 16798-1 (`ComfStand` 1, `CAT` 1, 2 and 3) with mixed-mode operation (`HVACmode` 2). `sampling_full_set` combines every value of each parameter with every EPW, so the plan has 3 x 2 = 6 simulations.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.

Variables created in this cell:
- `available_parameters`: reused in subsequent workflow blocks.
- `predef_parameters`: reused in subsequent workflow blocks.
- `plan_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `available_parameters`, `predef_parameters`, `plan_df`.

In [ ]:
available_parameters = parametric.get_available_parameters()
print("Available predefined parameters:", available_parameters)

predef_parameters = {
	"ComfStand": [1],
	"HVACmode": [2],
	"CAT": [1, 2, 3],
	"CATcoolOffset": [0],
	"CATheatOffset": [0],
}
parametric.set_parameters(accis_params_dict=predef_parameters)
parametric.sampling_full_set()
parametric.set_problem()

plan_df = parametric.parameters_values_df.copy()
print("Plan rows:", len(plan_df))
print(plan_df.head())


## ACCIM predefined parametric: estimate case count from plan and EPW multipliers

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

Variables created in this cell:
- `epw_multiplier`: reused in subsequent workflow blocks.
- `idf_multiplier`: reused in subsequent workflow blocks.
- `estimated_simulations`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
epw_multiplier = 1 if "epw" in plan_df.columns else len(epw_list)
idf_multiplier = 1 if "idf" in plan_df.columns else 1
estimated_simulations = len(plan_df) * epw_multiplier * idf_multiplier

print("EPW multiplier:", epw_multiplier)
print("IDF multiplier:", idf_multiplier)
print("Estimated simulation cases:", estimated_simulations)


## ACCIM predefined parametric: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

The simulation folders are kept, with only their CSV and ERR files (`sim_files_extensions`), because notebook 09 builds hourly and monthly tables from these CSV files.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [ ]:

outputs = parametric.run_parametric_simulation(
    out_dir=RESULTS_DIRNAME,
    df=plan_df,
    processes=PROCESSES,
    keep_input=True,
    keep_dirs=True,
    sim_files_extensions=[".csv", ".err"],
    sim_files_policy="keep",
)
print("Rows:", len(outputs))
print(outputs.head())


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## ACCIM predefined parametric: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [ ]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


## ACCIM predefined parametric: remove generated artifacts from this run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted: notebooks 09 and 10 read these results. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
